# Step 10: Model Input Preparation

Prepare segmented EEG data for model training with proper normalization and leakage prevention.

Rules:
- Load segments from split metadata
- Normalize using TRAINING data statistics only (channel-wise z-score)
- Apply same statistics to validation and test
- Primary classes: HC=0, AD=1, FTD=2
- MCI/UNKNOWN excluded from primary model
- Verify shapes, no NaN/Inf, no leakage
- Save X.npy, y.npy, and matching metadata

In [1]:
%pip install --force-reinstall "numpy==1.24.4" "pandas==1.5.3" -q

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
daal4py 2021.6.0 requires daal==2021.4.0, which is not installed.
numba 0.55.1 requires numpy<1.22,>=1.18, but you have numpy 1.24.4 which is incompatible.
conda-repo-cli 1.0.20 requires clyent==1.2.1, but you have clyent 1.2.2 which is incompatible.
conda-repo-cli 1.0.20 requires nbformat==5.4.0, but you have nbformat 5.5.0 which is incompatible.
conda-repo-cli 1.0.20 requires python-dateutil==2.8.2, but you have python-dateutil 2.9.0.post0 which is incompatible.
conda-repo-cli 1.0.20 requires requests==2.28.1, but you have requests 2.32.5 which is incompatible.
botocore 1.27.28 requires urllib3<1.27,>=1.25.4, but you have urllib3 2.6.3 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 80)
pd.set_option('display.max_colwidth', 200)

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name != 'NOTEBOOK':
    NOTEBOOK_DIR = Path('.').resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == 'NOTEBOOK' else NOTEBOOK_DIR
PROCESSED_DIR = PROJECT_ROOT / 'processed_data'
SPLITS_DIR = PROCESSED_DIR / 'splits'
MODEL_INPUT_DIR = PROCESSED_DIR / 'model_input'

MODEL_INPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_CHANNELS = [
    'Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8',
    'T3', 'C3', 'Cz', 'C4', 'T4', 'T5', 'P3',
    'Pz', 'P4', 'T6', 'O1', 'O2'
]
N_CHANNELS = 19
N_SAMPLES = 7500
TARGET_SFREQ = 250.0
SEG_SHAPE = (N_CHANNELS, N_SAMPLES)

LABEL_MAP = {'HC': 0, 'AD': 1, 'FTD': 2}
PRIMARY_LABELS = ['HC', 'AD', 'FTD']

print(f'Target shape per segment: {SEG_SHAPE}')
print(f'Label mapping: {LABEL_MAP}')

Target shape per segment: (19, 7500)
Label mapping: {'HC': 0, 'AD': 1, 'FTD': 2}


## 1. Load Split Metadata

In [3]:
train_meta = pd.read_csv(SPLITS_DIR / 'train_metadata.csv')
val_meta = pd.read_csv(SPLITS_DIR / 'validation_metadata.csv')
test_meta = pd.read_csv(SPLITS_DIR / 'test_metadata.csv')

print(f'Train segments: {len(train_meta)}')
print(f'Val segments: {len(val_meta)}')
print(f'Test segments: {len(test_meta)}')
print(f'Total: {len(train_meta) + len(val_meta) + len(test_meta)}')

for name, df in [('Train', train_meta), ('Val', val_meta), ('Test', test_meta)]:
    print(f'{name}: {df["standard_label"].value_counts().to_dict()}')

Train segments: 4505
Val segments: 763
Test segments: 969
Total: 6237
Train: {'AD': 1879, 'HC': 1518, 'FTD': 1108}
Val: {'AD': 273, 'FTD': 248, 'HC': 242}
Test: {'AD': 531, 'HC': 258, 'FTD': 180}


## 2. Load EEG Segments and Verify

In [4]:
def load_segments(meta_df, split_name):
    """Load all segments from metadata, verify shape/quality."""
    n_segments = len(meta_df)
    X = np.zeros((n_segments, N_CHANNELS, N_SAMPLES), dtype=np.float32)
    y = np.zeros(n_segments, dtype=np.int64)
    valid_indices = []
    errors = []
    
    for idx, row in meta_df.iterrows():
        seg_file = Path(row['segment_file'])
        label = row['standard_label']
        
        if label not in PRIMARY_LABELS:
            errors.append(f'idx {idx}: non-primary label {label}')
            continue
        
        if not seg_file.exists():
            errors.append(f'idx {idx}: file not found {seg_file}')
            continue
        
        try:
            seg_data = np.load(seg_file)
            
            if seg_data.shape != SEG_SHAPE:
                errors.append(f'idx {idx}: shape {seg_data.shape} != {SEG_SHAPE}')
                continue
            
            if np.any(np.isnan(seg_data)) or np.any(np.isinf(seg_data)):
                errors.append(f'idx {idx}: NaN/Inf values')
                continue
            
            X[idx] = seg_data.astype(np.float32)
            y[idx] = LABEL_MAP[label]
            valid_indices.append(idx)
            
        except Exception as exc:
            errors.append(f'idx {idx}: {exc}')
    
    X = X[valid_indices]
    y = y[valid_indices]
    meta_clean = meta_df.iloc[valid_indices].reset_index(drop=True)
    
    print(f'{split_name}: loaded {len(valid_indices)}/{n_segments} valid segments')
    if errors:
        print(f'  Errors ({len(errors)}): {errors[:5]}')
    
    return X, y, meta_clean, errors

X_train, y_train, train_meta_clean, train_errors = load_segments(train_meta, 'Train')
X_val, y_val, val_meta_clean, val_errors = load_segments(val_meta, 'Validation')
X_test, y_test, test_meta_clean, test_errors = load_segments(test_meta, 'Test')

all_errors = train_errors + val_errors + test_errors
if all_errors:
    print(f'\nTotal errors: {len(all_errors)}')
    for e in all_errors[:10]:
        print(f'  {e}')
else:
    print('\nAll segments loaded successfully!')

Train: loaded 4505/4505 valid segments
Validation: loaded 763/763 valid segments
Test: loaded 969/969 valid segments

All segments loaded successfully!


## 3. Compute Normalization Statistics from Training Data Only

In [5]:
print('=== COMPUTING NORMALIZATION STATISTICS (TRAINING ONLY) ===')

train_mean = X_train.mean(axis=(0, 2))
train_std = X_train.std(axis=(0, 2))

print('Training mean per channel (µV):')
for ch, m in zip(TARGET_CHANNELS, train_mean):
    print(f'  {ch}: {m:.2f}')

print('\nTraining std per channel (µV):')
for ch, s in zip(TARGET_CHANNELS, train_std):
    print(f'  {ch}: {s:.2f}')

if np.any(train_std == 0):
    zero_ch = [TARGET_CHANNELS[i] for i, s in enumerate(train_std) if s == 0]
    print(f'WARNING: Zero std for channels: {zero_ch}')
    train_std = np.where(train_std == 0, 1.0, train_std)

norm_stats = {
    'mean': train_mean.tolist(),
    'std': train_std.tolist(),
    'channels': TARGET_CHANNELS,
    'n_training_samples': int(X_train.shape[0]),
    'method': 'channel_wise_zscore'
}

with open(MODEL_INPUT_DIR / 'normalization_stats.json', 'w') as f:
    json.dump(norm_stats, f, indent=2)
print(f'\nSaved normalization stats to {MODEL_INPUT_DIR / "normalization_stats.json"}')

=== COMPUTING NORMALIZATION STATISTICS (TRAINING ONLY) ===
Training mean per channel (µV):
  Fp1: 0.00
  Fp2: 0.00
  F7: -0.00
  F3: 0.00
  Fz: 0.00
  F4: 0.00
  F8: -0.00
  T3: -0.00
  C3: -0.00
  Cz: 0.00
  C4: -0.00
  T4: -0.00
  T5: -0.00
  P3: -0.00
  Pz: -0.00
  P4: -0.00
  T6: -0.00
  O1: -0.00
  O2: -0.00

Training std per channel (µV):
  Fp1: 0.00
  Fp2: 0.00
  F7: 0.00
  F3: 0.00
  Fz: 0.00
  F4: 0.00
  F8: 0.00
  T3: 0.00
  C3: 0.00
  Cz: 0.00
  C4: 0.00
  T4: 0.00
  T5: 0.00
  P3: 0.00
  Pz: 0.00
  P4: 0.00
  T6: 0.00
  O1: 0.00
  O2: 0.00

Saved normalization stats to /home/admin1/Desktop/SIH_2026_ALL/processed_data/model_input/normalization_stats.json


## 4. Apply Normalization to All Splits

In [6]:
def normalize(X, mean, std):
    return (X - mean[None, :, None]) / std[None, :, None]

X_train_norm = normalize(X_train, train_mean, train_std)
X_val_norm = normalize(X_val, train_mean, train_std)
X_test_norm = normalize(X_test, train_mean, train_std)

print('Normalized shapes:')
print(f'  X_train: {X_train_norm.shape}')
print(f'  X_val: {X_val_norm.shape}')
print(f'  X_test: {X_test_norm.shape}')

print('\nVerification (train should be ~0 mean, ~1 std):')
print(f'  Train mean: {X_train_norm.mean():.4f}, std: {X_train_norm.std():.4f}')
print(f'  Val mean: {X_val_norm.mean():.4f}, std: {X_val_norm.std():.4f}')
print(f'  Test mean: {X_test_norm.mean():.4f}, std: {X_test_norm.std():.4f}')

for i, ch in enumerate(TARGET_CHANNELS):
    tr_m = X_train_norm[:, i, :].mean()
    tr_s = X_train_norm[:, i, :].std()
    if abs(tr_m) > 0.01 or abs(tr_s - 1.0) > 0.01:
        print(f'  {ch}: mean={tr_m:.4f}, std={tr_s:.4f}')

Normalized shapes:
  X_train: (4505, 19, 7500)
  X_val: (763, 19, 7500)
  X_test: (969, 19, 7500)

Verification (train should be ~0 mean, ~1 std):
  Train mean: 0.0000, std: 1.0000
  Val mean: 0.0001, std: 0.9254
  Test mean: 0.0000, std: 1.4313


## 5. Quality Checks

In [7]:
print('=== QUALITY CHECKS ===')

def check_quality(X, y, meta, split_name):
    issues = []
    
    if X.shape[1:] != SEG_SHAPE:
        issues.append(f'Shape mismatch: {X.shape[1:]} != {SEG_SHAPE}')
    
    if np.any(np.isnan(X)):
        issues.append('Contains NaN')
    if np.any(np.isinf(X)):
        issues.append('Contains Inf')
    
    unique_labels = np.unique(y)
    if not set(unique_labels).issubset({0, 1, 2}):
        issues.append(f'Invalid labels: {unique_labels}')
    
    if len(X) != len(y) or len(X) != len(meta):
        issues.append('Length mismatch between X, y, meta')
    
    # Verify metadata order matches X/y
    for idx, row in meta.iterrows():
        expected_label = LABEL_MAP.get(row['standard_label'])
        if expected_label is not None and y[idx] != expected_label:
            issues.append(f'Label mismatch at idx {idx}: meta={row["standard_label"]}, y={y[idx]}')
            break
    
    if issues:
        print(f'{split_name} ISSUES:')
        for issue in issues:
            print(f'  - {issue}')
    else:
        print(f'{split_name}: All checks passed ✓')
    
    return len(issues) == 0

check_quality(X_train_norm, y_train, train_meta_clean, 'Train')
check_quality(X_val_norm, y_val, val_meta_clean, 'Validation')
check_quality(X_test_norm, y_test, test_meta_clean, 'Test')

# Leakage check: no subject overlap
train_keys = set(train_meta_clean['subject_key'])
val_keys = set(val_meta_clean['subject_key'])
test_keys = set(test_meta_clean['subject_key'])

assert len(train_keys & val_keys) == 0, 'Train-Val subject overlap!'
assert len(train_keys & test_keys) == 0, 'Train-Test subject overlap!'
assert len(val_keys & test_keys) == 0, 'Val-Test subject overlap!'
print('\nNo subject leakage between splits ✓')

=== QUALITY CHECKS ===
Train: All checks passed ✓
Validation: All checks passed ✓
Test: All checks passed ✓

No subject leakage between splits ✓


## 6. Save Model Inputs

In [8]:
np.save(MODEL_INPUT_DIR / 'X_train.npy', X_train_norm)
np.save(MODEL_INPUT_DIR / 'y_train.npy', y_train)
np.save(MODEL_INPUT_DIR / 'X_validation.npy', X_val_norm)
np.save(MODEL_INPUT_DIR / 'y_validation.npy', y_val)
np.save(MODEL_INPUT_DIR / 'X_test.npy', X_test_norm)
np.save(MODEL_INPUT_DIR / 'y_test.npy', y_test)

train_meta_clean.to_csv(MODEL_INPUT_DIR / 'train_model_metadata.csv', index=False)
val_meta_clean.to_csv(MODEL_INPUT_DIR / 'validation_model_metadata.csv', index=False)
test_meta_clean.to_csv(MODEL_INPUT_DIR / 'test_model_metadata.csv', index=False)

print('Saved model inputs:')
print(f'  X_train.npy: {X_train_norm.shape}')
print(f'  y_train.npy: {y_train.shape}')
print(f'  X_validation.npy: {X_val_norm.shape}')
print(f'  y_validation.npy: {y_val.shape}')
print(f'  X_test.npy: {X_test_norm.shape}')
print(f'  y_test.npy: {y_test.shape}')
print(f'  train_model_metadata.csv: {len(train_meta_clean)}')
print(f'  validation_model_metadata.csv: {len(val_meta_clean)}')
print(f'  test_model_metadata.csv: {len(test_meta_clean)}')

Saved model inputs:
  X_train.npy: (4505, 19, 7500)
  y_train.npy: (4505,)
  X_validation.npy: (763, 19, 7500)
  y_validation.npy: (763,)
  X_test.npy: (969, 19, 7500)
  y_test.npy: (969,)
  train_model_metadata.csv: 4505
  validation_model_metadata.csv: 763
  test_model_metadata.csv: 969


## 7. Class Distribution Report

In [9]:
print('=== CLASS DISTRIBUTION ===')

for name, y_arr, meta_df in [
    ('Train', y_train, train_meta_clean),
    ('Validation', y_val, val_meta_clean),
    ('Test', y_test, test_meta_clean)
]:  
    unique, counts = np.unique(y_arr, return_counts=True)
    print(f'{name} segments:')
    for u, c in zip(unique, counts):
        label_name = {v: k for k, v in LABEL_MAP.items()}[u]
        print(f'  {label_name} ({u}): {c} ({c/len(y_arr)*100:.1f}%)')
    
    # Subject-level
    subj_labels = meta_df.groupby('subject_key')['standard_label'].first()
    subj_counts = subj_labels.value_counts()
    print(f'{name} subjects:')
    for label, count in subj_counts.items():
        print(f'  {label}: {count}')
    print()

=== CLASS DISTRIBUTION ===
Train segments:
  HC (0): 1518 (33.7%)
  AD (1): 1879 (41.7%)
  FTD (2): 1108 (24.6%)
Train subjects:
  AD: 25
  HC: 19
  FTD: 12

Validation segments:
  HC (0): 242 (31.7%)
  AD (1): 273 (35.8%)
  FTD (2): 248 (32.5%)
Validation subjects:
  AD: 5
  HC: 4
  FTD: 3

Test segments:
  HC (0): 258 (26.6%)
  AD (1): 531 (54.8%)
  FTD (2): 180 (18.6%)
Test subjects:
  AD: 6
  HC: 4
  FTD: 2



## 8. Final Verification and Report

In [10]:
print('='*60)
print('FINAL MODEL INPUT REPORT')
print('='*60)

print(f'X_train shape: {X_train_norm.shape}')
print(f'X_validation shape: {X_val_norm.shape}')
print(f'X_test shape: {X_test_norm.shape}')

print(f'\ny_train shape: {y_train.shape}')
print(f'y_validation shape: {y_val.shape}')
print(f'y_test shape: {y_test.shape}')

print(f'\nTotal training segments: {len(X_train_norm)}')
print(f'Total validation segments: {len(X_val_norm)}')
print(f'Total test segments: {len(X_test_norm)}')

print(f'\nTraining subjects: {len(train_meta_clean["subject_key"].unique())}')
print(f'Validation subjects: {len(val_meta_clean["subject_key"].unique())}')
print(f'Test subjects: {len(test_meta_clean["subject_key"].unique())}')

print(f'\nNormalization: channel-wise z-score using training data only')
print(f'Stats saved: {MODEL_INPUT_DIR / "normalization_stats.json"}')

print('\nMODEL INPUT PREPARATION COMPLETE — READY FOR BASELINE MODEL')

FINAL MODEL INPUT REPORT
X_train shape: (4505, 19, 7500)
X_validation shape: (763, 19, 7500)
X_test shape: (969, 19, 7500)

y_train shape: (4505,)
y_validation shape: (763,)
y_test shape: (969,)

Total training segments: 4505
Total validation segments: 763
Total test segments: 969

Training subjects: 56
Validation subjects: 12
Test subjects: 12

Normalization: channel-wise z-score using training data only
Stats saved: /home/admin1/Desktop/SIH_2026_ALL/processed_data/model_input/normalization_stats.json

MODEL INPUT PREPARATION COMPLETE — READY FOR BASELINE MODEL
